# Fase 5: Anatomía espectral y compresión óptima

Dissección de la estructura interna de BERT a través del espectro de valores singulares
de sus 72 capas lineales. Este análisis revela **por qué** ciertos componentes toleran
mejor la compresión y conduce a una estrategia de compresión adaptativa que supera
a la compresión uniforme.

**Parte A** — Análisis espectral (solo pesos, sin inferencia): fingerprints, heatmaps, diagrama de arquitectura

**Parte B** — Análisis de rendimiento (con inferencia): vulnerabilidad por emoción, frontera de Pareto

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
import seaborn as sns
from matplotlib.patches import FancyBboxPatch
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer

from src.data import load_goemotions
from src.data.dataset import NUM_LABELS, EMOTION_NAMES
from src.utils import compute_metrics
from src.compression import (
    apply_svd_compression,
    compute_singular_value_energy,
    compute_adaptive_ranks,
    count_parameters,
    get_compression_ratio,
    get_target_layer_names,
    filter_layer_names,
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 200
plt.rcParams['font.size'] = 11
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

results_dir = os.path.join(PROJECT_ROOT, 'results')
os.makedirs(results_dir, exist_ok=True)

In [ ]:
# Cargar modelo y calcular espectro SVD de las 72 capas
model_path = os.path.join(PROJECT_ROOT, 'results', 'bert-goemotions-final')
baseline_model = AutoModelForSequenceClassification.from_pretrained(
    model_path, num_labels=NUM_LABELS, problem_type='multi_label_classification',
)
baseline_model.eval()
baseline_model.to(device)

print('Calculando espectro SVD de 72 capas...')
energy_info = compute_singular_value_energy(baseline_model)
print(f'Capas analizadas: {len(energy_info)}')

In [ ]:
# Organizar datos espectrales en una estructura manejable
COMP_PATTERNS = {
    'Query (Q)': 'attention.self.query',
    'Key (K)': 'attention.self.key',
    'Value (V)': 'attention.self.value',
    'Attn Output': 'attention.output.dense',
    'FFN Inter': 'intermediate.dense',
    'FFN Output': 'output.dense',
}
COMP_LABELS = list(COMP_PATTERNS.keys())
COMP_COLORS = ['#e74c3c', '#e67e22', '#f1c40f', '#2ecc71', '#3498db', '#9b59b6']

def get_component(name):
    """Determina el tipo de componente a partir del nombre de la capa."""
    if 'attention.self.query' in name: return 'Query (Q)'
    if 'attention.self.key' in name: return 'Key (K)'
    if 'attention.self.value' in name: return 'Value (V)'
    if 'attention.output.dense' in name: return 'Attn Output'
    if 'intermediate.dense' in name: return 'FFN Inter'
    if 'output.dense' in name and 'attention' not in name: return 'FFN Output'
    return None

def get_layer_idx(name):
    return int(name.split('.')[3])

# Organizar espectros por componente y capa
spectra = {comp: [] for comp in COMP_LABELS}
rank_matrix = np.zeros((12, 6))  # layer x component
rank95_matrix = np.zeros((12, 6))
rank90_matrix = np.zeros((12, 6))
rank99_matrix = np.zeros((12, 6))

for name, info in energy_info.items():
    comp = get_component(name)
    layer_idx = get_layer_idx(name)
    comp_idx = COMP_LABELS.index(comp)

    sv = info['singular_values'].numpy()
    spectra[comp].append(sv)

    rank90_matrix[layer_idx, comp_idx] = info['rank_for_90']
    rank95_matrix[layer_idx, comp_idx] = info['rank_for_95']
    rank99_matrix[layer_idx, comp_idx] = info['rank_for_99']

print('Datos espectrales organizados:')
for comp in COMP_LABELS:
    n = len(spectra[comp])
    n_sv = len(spectra[comp][0])
    print(f'  {comp:<15s}: {n} capas, {n_sv} valores singulares cada una')

---
## Parte A: Análisis espectral (sin inferencia)

### 1. Fingerprint espectral por componente

Cada tipo de componente tiene una estructura matemática diferente.
Las curvas de decaimiento de valores singulares revelan su **compresibilidad intrínseca**:
un decaimiento rápido indica que la información se concentra en pocos vectores singulares
(alta compresibilidad), mientras que un decaimiento lento indica información distribuida
(baja compresibilidad).

In [ ]:
fig, (ax_main, ax_zoom) = plt.subplots(1, 2, figsize=(18, 7),
                                        gridspec_kw={'width_ratios': [2, 1]})

for i, comp in enumerate(COMP_LABELS):
    curves = np.array(spectra[comp])
    # Normalizar cada curva por su mayor valor singular
    curves_norm = curves / curves[:, 0:1]
    mean = curves_norm.mean(axis=0)
    std = curves_norm.std(axis=0)
    x = np.arange(1, len(mean) + 1)

    ax_main.plot(x, mean, color=COMP_COLORS[i], label=comp, linewidth=2.5)
    ax_main.fill_between(x, mean - std, mean + std,
                         color=COMP_COLORS[i], alpha=0.12)

    # Zoom: primeros 200 valores singulares
    ax_zoom.plot(x[:200], mean[:200], color=COMP_COLORS[i], linewidth=2.5)
    ax_zoom.fill_between(x[:200], (mean - std)[:200], (mean + std)[:200],
                         color=COMP_COLORS[i], alpha=0.12)

# Líneas de referencia para rangos típicos
for ax in [ax_main]:
    for rank_val, ls in [(64, ':'), (128, '--'), (256, '-.')]: 
        ax.axvline(x=rank_val, color='gray', linestyle=ls, alpha=0.4, linewidth=1)
        ax.text(rank_val + 3, 0.92, f'r={rank_val}', fontsize=8, color='gray', alpha=0.7)

ax_main.set_xlabel('Índice del valor singular', fontsize=13)
ax_main.set_ylabel('Valor singular normalizado ($\\sigma_i / \\sigma_1$)', fontsize=13)
ax_main.set_title('Fingerprint espectral por componente\n(media $\\pm$ 1 std sobre 12 capas)',
                   fontsize=14, fontweight='bold')
ax_main.legend(fontsize=11, loc='upper right')
ax_main.set_ylim(-0.02, 1.02)
ax_main.set_xlim(0, 768)

ax_zoom.set_xlabel('Índice del valor singular', fontsize=12)
ax_zoom.set_ylabel('Valor singular normalizado', fontsize=12)
ax_zoom.set_title('Zoom: primeros 200', fontsize=12, fontweight='bold')
ax_zoom.set_ylim(-0.02, 1.02)
ax_zoom.set_xlim(0, 200)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'spectral_fingerprint.png'), bbox_inches='tight')
plt.show()
print('Guardada: spectral_fingerprint.png')

### 2. Mapa de compresibilidad — Las 72 capas de BERT

Cada celda muestra el **rango mínimo necesario para retener el 95% de la energía
de los valores singulares**. Un valor bajo (verde) indica alta compresibilidad;
un valor alto (rojo) indica que la capa es sensible a la compresión.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(22, 8))

for ax, matrix, threshold in zip(axes, [rank90_matrix, rank95_matrix, rank99_matrix],
                                       ['90%', '95%', '99%']):
    sns.heatmap(
        matrix, annot=True, fmt='.0f',
        cmap='RdYlGn_r',
        xticklabels=COMP_LABELS,
        yticklabels=[f'Capa {i}' for i in range(12)],
        linewidths=0.8, linecolor='white',
        cbar_kws={'label': 'Rango necesario', 'shrink': 0.8},
        ax=ax,
    )
    ax.set_title(f'Energía retenida: {threshold}', fontsize=13, fontweight='bold')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha='right')

fig.suptitle('Mapa de compresibilidad de BERT-base\n(rango mínimo por capa y componente)',
             fontsize=16, fontweight='bold', y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'compressibility_map.png'), bbox_inches='tight')
plt.show()
print('Guardada: compressibility_map.png')

### 3. Diagrama de arquitectura — Radiografía de BERT

Representación visual del encoder completo, con cada componente
coloreado según su compresibilidad. Como una **resonancia magnética del modelo**.

In [ ]:
fig, ax = plt.subplots(figsize=(16, 12))
ax.set_aspect('auto')
ax.axis('off')

# Layout
H = 0.65          # altura de cada bloque
GAP_Y = 0.18      # espacio vertical entre capas
W = 2.0            # ancho de cada bloque
GAP_X = 0.15      # espacio horizontal entre bloques
BLOCK_GAP = 0.8   # espacio entre atención y FFN

# Colormap
cmap = plt.cm.RdYlGn_r
vmin, vmax = rank95_matrix.min(), rank95_matrix.max()
norm = Normalize(vmin=vmin, vmax=vmax)

# Dibujar capas (de abajo arriba)
for layer_idx in range(12):
    y = layer_idx * (H + GAP_Y)

    # Label de capa
    ax.text(-0.8, y + H / 2, f'Capa {layer_idx}', ha='right', va='center',
            fontsize=10, fontweight='bold', family='monospace')

    # Atención: Q, K, V, Out (4 bloques)
    for j in range(4):
        x = j * (W + GAP_X)
        rank = rank95_matrix[layer_idx, j]
        color = cmap(norm(rank))

        rect = FancyBboxPatch((x, y), W, H,
                              boxstyle='round,pad=0.05',
                              facecolor=color, edgecolor='white',
                              linewidth=2, zorder=2)
        ax.add_patch(rect)
        comp_short = ['Q', 'K', 'V', 'Out'][j]
        ax.text(x + W / 2, y + H / 2, f'{comp_short}\n{int(rank)}',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if rank > (vmin + vmax) / 2 else 'black', zorder=3)

    # FFN: Inter, Out (2 bloques, más anchos visualmente)
    ffn_x_start = 4 * (W + GAP_X) + BLOCK_GAP
    ffn_w = 2.5  # más anchos para reflejar que tienen más parámetros
    for j in range(2):
        x = ffn_x_start + j * (ffn_w + GAP_X)
        comp_idx = j + 4
        rank = rank95_matrix[layer_idx, comp_idx]
        color = cmap(norm(rank))

        rect = FancyBboxPatch((x, y), ffn_w, H,
                              boxstyle='round,pad=0.05',
                              facecolor=color, edgecolor='white',
                              linewidth=2, zorder=2)
        ax.add_patch(rect)
        comp_short = ['Inter', 'Output'][j]
        ax.text(x + ffn_w / 2, y + H / 2, f'{comp_short}\n{int(rank)}',
                ha='center', va='center', fontsize=9, fontweight='bold',
                color='white' if rank > (vmin + vmax) / 2 else 'black', zorder=3)

    # Línea divisoria entre Atención y FFN
    div_x = 4 * (W + GAP_X) + BLOCK_GAP / 2 - 0.05
    ax.plot([div_x, div_x], [y - 0.02, y + H + 0.02],
            color='gray', linewidth=0.8, linestyle=':', alpha=0.5)

# Encabezados de bloque
top_y = 12 * (H + GAP_Y) + 0.15
attn_center = (3 * (W + GAP_X) + W) / 2
ffn_center = ffn_x_start + (ffn_w + GAP_X + ffn_w) / 2

ax.text(attn_center, top_y + 0.5, 'SELF-ATTENTION',
        ha='center', fontsize=16, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightblue', alpha=0.3))
ax.text(ffn_center, top_y + 0.5, 'FEED-FORWARD',
        ha='center', fontsize=16, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.3))

# Título
ax.text((attn_center + ffn_center) / 2, top_y + 1.5,
        'Anatomía de BERT-base: compresibilidad por capa y componente',
        ha='center', fontsize=18, fontweight='bold')
ax.text((attn_center + ffn_center) / 2, top_y + 1.0,
        'Rango mínimo para retener 95% de la energía singular (verde = muy compresible, rojo = sensible)',
        ha='center', fontsize=11, color='gray')

# Ajustar límites
ax.set_xlim(-1.5, ffn_x_start + 2 * (ffn_w + GAP_X) + 0.5)
ax.set_ylim(-0.5, top_y + 2.2)

# Colorbar
sm = ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=ax, shrink=0.4, pad=0.02, aspect=30)
cbar.set_label('Rango necesario (95% energía)', fontsize=12)

plt.savefig(os.path.join(results_dir, 'bert_architecture_anatomy.png'), bbox_inches='tight')
plt.show()
print('Guardada: bert_architecture_anatomy.png')

### 4. Curvas de energía acumulada

Para 3 capas representativas (temprana, media, tardía), mostramos cómo la energía
se acumula con cada valor singular. Las capas con curvas que suben rápido
son comprimibles con rangos bajos.

In [ ]:
selected_layers = [0, 5, 11]
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for ax, layer_idx in zip(axes, selected_layers):
    for i, comp in enumerate(COMP_LABELS):
        # Encontrar la capa correspondiente en energy_info
        pattern = COMP_PATTERNS[comp]
        for name, info in energy_info.items():
            if get_layer_idx(name) == layer_idx and get_component(name) == comp:
                cum_energy = info['cumulative_energy'].numpy()
                x = np.arange(1, len(cum_energy) + 1)
                ax.plot(x, cum_energy, color=COMP_COLORS[i], label=comp,
                        linewidth=2, alpha=0.9)
                break

    # Líneas de umbral
    for threshold, ls, alpha in [(0.90, ':', 0.4), (0.95, '--', 0.5), (0.99, '-', 0.3)]:
        ax.axhline(y=threshold, color='gray', linestyle=ls, alpha=alpha, linewidth=1)
        ax.text(750, threshold - 0.015, f'{int(threshold*100)}%',
                fontsize=8, color='gray', ha='right')

    ax.set_xlabel('Rango (k)', fontsize=12)
    ax.set_ylabel('Energía acumulada', fontsize=12)
    ax.set_title(f'Capa {layer_idx}', fontsize=14, fontweight='bold')
    ax.set_xlim(0, 400)
    ax.set_ylim(0.7, 1.005)
    if layer_idx == 0:
        ax.legend(fontsize=9, loc='lower right')

fig.suptitle('Curvas de energía acumulada por componente\n'
             '(rápida subida = alta compresibilidad)',
             fontsize=15, fontweight='bold', y=1.04)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'cumulative_energy_curves.png'), bbox_inches='tight')
plt.show()
print('Guardada: cumulative_energy_curves.png')

### 5. Compresión adaptativa — Asignación de rangos por umbral

En vez de un rango fijo para todas las capas, asignamos a cada capa el rango mínimo
que retiene un porcentaje de energía dado. Mostramos cómo cambia la distribución
de rangos según el umbral.

In [ ]:
thresholds = [0.80, 0.85, 0.90, 0.95, 0.99]
adaptive_configs = {}

for t in thresholds:
    ranks = compute_adaptive_ranks(energy_info, energy_threshold=t)
    adaptive_configs[t] = ranks
    total_params = sum(ranks.values())
    avg_rank = np.mean(list(ranks.values()))
    print(f'Umbral {t:.0%}: rango medio = {avg_rank:.1f}, '
          f'rango min = {min(ranks.values())}, rango max = {max(ranks.values())}')

In [ ]:
# Violin plots: distribución de rangos asignados por umbral
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 7),
                                gridspec_kw={'width_ratios': [1.3, 1]})

# --- Panel izquierdo: violines ---
violin_data = []
violin_labels = []
for t in thresholds:
    ranks = list(adaptive_configs[t].values())
    violin_data.append(ranks)
    violin_labels.append(f'{t:.0%}')

parts = ax1.violinplot(violin_data, positions=range(len(thresholds)),
                       showmeans=True, showmedians=True, showextrema=False)

colors_violin = ['#27ae60', '#2ecc71', '#f39c12', '#e67e22', '#e74c3c']
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(colors_violin[i])
    pc.set_alpha(0.7)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('navy')

# Superponer puntos individuales (jittered)
for i, ranks in enumerate(violin_data):
    jitter = np.random.RandomState(42).normal(0, 0.04, size=len(ranks))
    ax1.scatter(np.full(len(ranks), i) + jitter, ranks,
               c='black', s=8, alpha=0.4, zorder=3)

ax1.set_xticks(range(len(thresholds)))
ax1.set_xticklabels([f'{t:.0%}' for t in thresholds], fontsize=12)
ax1.set_xlabel('Umbral de energía', fontsize=13)
ax1.set_ylabel('Rango asignado', fontsize=13)
ax1.set_title('Distribución de rangos adaptativos', fontsize=14, fontweight='bold')

# Líneas de referencia para rangos uniformes
for r, ls in [(64, ':'), (128, '--'), (256, '-.')]:
    ax1.axhline(y=r, color='gray', linestyle=ls, alpha=0.4)
    ax1.text(len(thresholds) - 0.6, r + 3, f'r={r}', fontsize=8, color='gray')

# --- Panel derecho: rango medio por componente y umbral ---
comp_means = np.zeros((len(COMP_LABELS), len(thresholds)))
for j, t in enumerate(thresholds):
    for name, rank in adaptive_configs[t].items():
        comp = get_component(name)
        comp_idx = COMP_LABELS.index(comp)
        comp_means[comp_idx, j] += rank
    comp_means[:, j] /= 12  # media sobre 12 capas

for i, comp in enumerate(COMP_LABELS):
    ax2.plot(thresholds, comp_means[i, :], 'o-',
             color=COMP_COLORS[i], label=comp, linewidth=2.5, markersize=8)

ax2.set_xlabel('Umbral de energía', fontsize=13)
ax2.set_ylabel('Rango medio asignado', fontsize=13)
ax2.set_title('Rango medio por componente', fontsize=14, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xticks(thresholds)
ax2.set_xticklabels([f'{t:.0%}' for t in thresholds])

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'adaptive_rank_distribution.png'), bbox_inches='tight')
plt.show()
print('Guardada: adaptive_rank_distribution.png')

---
## Parte B: Análisis de rendimiento (con inferencia)

### 6. Vulnerabilidad por emoción bajo compresión

¿La compresión afecta a todas las emociones por igual?
Evaluamos el F1 por emoción a distintos niveles de compresión.

In [ ]:
# Cargar datos
_, _, test_ds, _, data_collator = load_goemotions()
print(f'Test set: {len(test_ds)} ejemplos')

eval_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_ROOT, 'results', 'tmp_eval'),
    per_device_eval_batch_size=64,
    fp16=(device == 'cuda'),
    report_to='none',
)

def evaluate_model(model):
    trainer = Trainer(
        model=model, args=eval_args,
        compute_metrics=compute_metrics, data_collator=data_collator,
    )
    return trainer.evaluate(test_ds)

In [ ]:
# Evaluar baseline + compresiones uniformes
UNIFORM_RANKS = [512, 384, 256, 128, 64]
all_per_emotion = {}  # name -> array of 28 F1 scores
all_f1_macro = {}     # name -> float
all_ratio = {}        # name -> float

# Baseline
print('Evaluando Baseline...')
metrics_bl = evaluate_model(baseline_model)
all_f1_macro['Baseline'] = metrics_bl['eval_f1_macro']
all_ratio['Baseline'] = 1.0
all_per_emotion['Baseline'] = np.array(
    [metrics_bl[f'eval_f1_label_{i}'] for i in range(NUM_LABELS)]
)
print(f'  F1 macro: {all_f1_macro["Baseline"]:.4f}')

# Compresión uniforme
for rank in UNIFORM_RANKS:
    name = f'Uniform r={rank}'
    print(f'Evaluando {name}...')
    model = apply_svd_compression(baseline_model, rank=rank)
    model.to(device)
    metrics = evaluate_model(model)
    all_f1_macro[name] = metrics['eval_f1_macro']
    all_ratio[name] = get_compression_ratio(baseline_model, model)
    all_per_emotion[name] = np.array(
        [metrics[f'eval_f1_label_{i}'] for i in range(NUM_LABELS)]
    )
    print(f'  F1 macro: {all_f1_macro[name]:.4f}, ratio: {all_ratio[name]:.2f}x')
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

print(f'\nModelos evaluados: {len(all_f1_macro)}')

In [ ]:
# Evaluar compresiones adaptativas
for t in thresholds:
    name = f'Adaptive {t:.0%}'
    print(f'Evaluando {name}...')
    ranks = adaptive_configs[t]
    model = apply_svd_compression(baseline_model, rank=ranks)
    model.to(device)
    metrics = evaluate_model(model)
    all_f1_macro[name] = metrics['eval_f1_macro']
    all_ratio[name] = get_compression_ratio(baseline_model, model)
    all_per_emotion[name] = np.array(
        [metrics[f'eval_f1_label_{i}'] for i in range(NUM_LABELS)]
    )
    print(f'  F1 macro: {all_f1_macro[name]:.4f}, ratio: {all_ratio[name]:.2f}x')
    del model
    if device == 'cuda':
        torch.cuda.empty_cache()

print(f'\nTotal modelos evaluados: {len(all_f1_macro)}')

In [ ]:
# Heatmap de vulnerabilidad por emoción
baseline_f1 = all_per_emotion['Baseline']
compare_keys = ['Baseline'] + [f'Uniform r={r}' for r in UNIFORM_RANKS]
f1_matrix = np.array([all_per_emotion[k] for k in compare_keys]).T  # (28, n_models)

# Ordenar emociones por vulnerabilidad (mayor drop en rank 64)
f1_drop = baseline_f1 - all_per_emotion['Uniform r=64']
sort_idx = np.argsort(-f1_drop)  # mayor drop primero
sorted_emotions = [EMOTION_NAMES[i] for i in sort_idx]
sorted_matrix = f1_matrix[sort_idx, :]

fig, ax = plt.subplots(figsize=(10, 13))
sns.heatmap(
    sorted_matrix, annot=True, fmt='.2f',
    cmap='RdYlGn', vmin=0, vmax=1,
    xticklabels=['Baseline'] + [f'r={r}' for r in UNIFORM_RANKS],
    yticklabels=sorted_emotions,
    linewidths=0.5, linecolor='white',
    cbar_kws={'label': 'F1 score', 'shrink': 0.6},
    ax=ax,
)

# Marcar las emociones más vulnerables
for i in range(5):
    ax.get_yticklabels()[i].set_fontweight('bold')
    ax.get_yticklabels()[i].set_color('#c0392b')

ax.set_title('Vulnerabilidad por emoción bajo compresión SVD uniforme\n'
             '(ordenado por mayor degradación, top 5 en rojo)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Nivel de compresión', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'emotion_vulnerability_heatmap.png'), bbox_inches='tight')
plt.show()
print('Guardada: emotion_vulnerability_heatmap.png')

In [ ]:
# Radar chart de vulnerabilidad por emoción
fig, ax = plt.subplots(figsize=(12, 12), subplot_kw=dict(projection='polar'))

angles = np.linspace(0, 2 * np.pi, NUM_LABELS, endpoint=False).tolist()
angles += angles[:1]  # cerrar el polígono

radar_configs = [
    ('Baseline', '#2ecc71', 2.5, '-', 0.12),
    ('Uniform r=256', '#3498db', 2.0, '--', 0.06),
    ('Uniform r=128', '#e67e22', 2.0, '--', 0.06),
    ('Uniform r=64', '#e74c3c', 2.5, '-', 0.08),
]

for name, color, lw, ls, fill_alpha in radar_configs:
    values = all_per_emotion[name].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', color=color, linewidth=lw, linestyle=ls,
            label=name, markersize=3)
    ax.fill(angles, values, color=color, alpha=fill_alpha)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(EMOTION_NAMES, fontsize=8)
ax.set_ylim(0, 1)
ax.set_rticks([0.2, 0.4, 0.6, 0.8])
ax.set_rlabel_position(30)
ax.set_title('F1 por emoción bajo compresión SVD\n',
             fontsize=16, fontweight='bold', pad=30)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'emotion_radar.png'), bbox_inches='tight')
plt.show()
print('Guardada: emotion_radar.png')

### 7. Frontera de Pareto — Uniforme vs. Adaptativa

Comparamos todas las estrategias en un único gráfico:
- **Compresión uniforme**: mismo rango para todas las capas
- **Compresión adaptativa**: rango por capa según umbral de energía

La frontera de Pareto conecta las configuraciones no dominadas
(mejores trade-offs posibles).

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))

baseline_f1_macro = all_f1_macro['Baseline']

# Plot uniform strategies
uniform_ratios = [all_ratio[f'Uniform r={r}'] for r in UNIFORM_RANKS]
uniform_retentions = [all_f1_macro[f'Uniform r={r}'] / baseline_f1_macro * 100
                      for r in UNIFORM_RANKS]
ax.scatter(uniform_ratios, uniform_retentions, c='#3498db', s=150, zorder=5,
           edgecolors='white', linewidth=2, label='Uniforme')
for r, x, y in zip(UNIFORM_RANKS, uniform_ratios, uniform_retentions):
    ax.annotate(f'r={r}', (x, y), textcoords='offset points',
                xytext=(10, 8), fontsize=10, color='#2c3e50', fontweight='bold')

# Plot adaptive strategies
adaptive_ratios = [all_ratio[f'Adaptive {t:.0%}'] for t in thresholds]
adaptive_retentions = [all_f1_macro[f'Adaptive {t:.0%}'] / baseline_f1_macro * 100
                       for t in thresholds]
ax.scatter(adaptive_ratios, adaptive_retentions, c='#e74c3c', s=150, zorder=5,
           edgecolors='white', linewidth=2, marker='D', label='Adaptativa')
for t, x, y in zip(thresholds, adaptive_ratios, adaptive_retentions):
    ax.annotate(f'{t:.0%}', (x, y), textcoords='offset points',
                xytext=(10, -12), fontsize=10, color='#c0392b', fontweight='bold')

# Baseline point
ax.scatter([1.0], [100.0], c='#2ecc71', s=200, zorder=6,
           edgecolors='white', linewidth=2, marker='*', label='Baseline')
ax.annotate('Baseline', (1.0, 100.0), textcoords='offset points',
            xytext=(12, 5), fontsize=11, fontweight='bold', color='#27ae60')

# Compute and plot Pareto frontier
all_points = [(1.0, 100.0)]  # baseline
all_points += list(zip(uniform_ratios, uniform_retentions))
all_points += list(zip(adaptive_ratios, adaptive_retentions))
all_points.sort(key=lambda p: p[0])

pareto_points = [all_points[0]]
for p in all_points[1:]:
    if p[1] >= pareto_points[-1][1] - 0.001:  # non-dominated
        pareto_points.append(p)
    elif p[0] > pareto_points[-1][0]:  # higher compression but check dominance
        pareto_points.append(p)

# Simple Pareto: sort by ratio, keep those with retention >= max seen so far from right
all_pts_sorted = sorted(all_points, key=lambda p: p[0])
pareto = [all_pts_sorted[0]]
for pt in all_pts_sorted[1:]:
    if pt[1] > pareto[-1][1] - 5:  # within reasonable range
        pareto.append(pt)
pareto_x = [p[0] for p in pareto]
pareto_y = [p[1] for p in pareto]
ax.plot(pareto_x, pareto_y, '--', color='gray', alpha=0.5, linewidth=1.5, zorder=1)

# Shading the "better" region
ax.fill_between([0.9, max(uniform_ratios + adaptive_ratios) + 0.1],
                [95, 95], [105, 105], alpha=0.05, color='green',
                label='Zona ideal (>95% retención)')

ax.set_xlabel('Ratio de compresión (x)', fontsize=14)
ax.set_ylabel('Retención de F1 macro (%)', fontsize=14)
ax.set_title('Frontera de Pareto: Compresión uniforme vs. adaptativa\n'
             '(más arriba y a la derecha = mejor)',
             fontsize=15, fontweight='bold')
ax.legend(fontsize=12, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_ylim(min(uniform_retentions + adaptive_retentions) - 5, 102)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'pareto_frontier.png'), bbox_inches='tight')
plt.show()
print('Guardada: pareto_frontier.png')

### 8. Lollipop: emociones más vulnerables

Cuantificamos exactamente cuánto F1 pierde cada emoción bajo compresión agresiva (rank 64).
Las emociones raras y semánticamente sutiles son las primeras en colapsar.

In [ ]:
# Lollipop chart: F1 drop per emotion (baseline - rank 64)
drops = all_per_emotion['Baseline'] - all_per_emotion['Uniform r=64']
sort_idx = np.argsort(drops)  # menor a mayor drop
sorted_drops = drops[sort_idx]
sorted_names = [EMOTION_NAMES[i] for i in sort_idx]
sorted_baseline = all_per_emotion['Baseline'][sort_idx]

fig, ax = plt.subplots(figsize=(10, 12))

# Colorear por magnitud del drop
colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(sorted_drops)))

for i, (name, drop, bl) in enumerate(zip(sorted_names, sorted_drops, sorted_baseline)):
    # Línea del lollipop
    ax.plot([bl - drop, bl], [i, i], color=colors[i], linewidth=2.5, zorder=2)
    # Punto baseline (verde)
    ax.scatter(bl, i, color='#2ecc71', s=60, zorder=3, edgecolors='white', linewidth=1)
    # Punto comprimido (rojo)
    ax.scatter(bl - drop, i, color=colors[i], s=60, zorder=3, edgecolors='white', linewidth=1)

ax.set_yticks(range(len(sorted_names)))
ax.set_yticklabels(sorted_names, fontsize=10)
ax.set_xlabel('F1 score', fontsize=13)
ax.set_title('Impacto de la compresión SVD (rank 64) por emoción\n'
             'Punto verde = baseline, punto coloreado = comprimido',
             fontsize=14, fontweight='bold')
ax.set_xlim(-0.05, 1.0)

# Leyenda
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#2ecc71',
           markersize=10, label='Baseline'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor='#e74c3c',
           markersize=10, label='Rank 64'),
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(results_dir, 'emotion_lollipop.png'), bbox_inches='tight')
plt.show()
print('Guardada: emotion_lollipop.png')

---
## 9. Exportar resultados y síntesis

In [ ]:
# Exportar resultados a CSV
rows = []
for name in all_f1_macro:
    rows.append({
        'strategy': name,
        'f1_macro': all_f1_macro[name],
        'compression_ratio': all_ratio[name],
        'f1_retention': all_f1_macro[name] / baseline_f1_macro * 100,
    })
df_results = pd.DataFrame(rows)
csv_path = os.path.join(results_dir, 'spectral_analysis_results.csv')
df_results.to_csv(csv_path, index=False)
print(f'Guardado: {csv_path}')
print(df_results.to_string(index=False))

# Exportar F1 per emotion
emotion_rows = []
for name in all_per_emotion:
    for i, emo in enumerate(EMOTION_NAMES):
        emotion_rows.append({
            'strategy': name,
            'emotion': emo,
            'f1': all_per_emotion[name][i],
        })
df_emotions = pd.DataFrame(emotion_rows)
emo_csv_path = os.path.join(results_dir, 'per_emotion_results.csv')
df_emotions.to_csv(emo_csv_path, index=False)
print(f'\nGuardado: {emo_csv_path} ({len(df_emotions)} filas)')

In [ ]:
figures = [
    'spectral_fingerprint.png',
    'compressibility_map.png',
    'bert_architecture_anatomy.png',
    'cumulative_energy_curves.png',
    'adaptive_rank_distribution.png',
    'emotion_vulnerability_heatmap.png',
    'emotion_radar.png',
    'pareto_frontier.png',
    'emotion_lollipop.png',
]

print('=' * 65)
print('RESUMEN — Fase 5: Anatomía espectral y compresión óptima')
print('=' * 65)
print(f'\nBaseline F1 macro: {baseline_f1_macro:.4f}')
print(f'Modelos evaluados: {len(all_f1_macro)}')
print(f'  - 1 baseline')
print(f'  - {len(UNIFORM_RANKS)} compresiones uniformes')
print(f'  - {len(thresholds)} compresiones adaptativas')
print(f'\nMejor adaptativa: ', end='')
best_adaptive = max(
    [(t, all_f1_macro[f'Adaptive {t:.0%}'], all_ratio[f'Adaptive {t:.0%}'])
     for t in thresholds],
    key=lambda x: x[1]
)
print(f'{best_adaptive[0]:.0%} energía -> '
      f'F1={best_adaptive[1]:.4f}, ratio={best_adaptive[2]:.2f}x')
print(f'\nEmociones más vulnerables (mayor drop con rank 64):')
for i in range(5):
    idx = np.argsort(-drops)[i]
    print(f'  {EMOTION_NAMES[idx]:<15s}: {drops[idx]:+.4f} '
          f'({all_per_emotion["Baseline"][idx]:.3f} -> '
          f'{all_per_emotion["Uniform r=64"][idx]:.3f})')
print(f'\nFiguras generadas ({len(figures)}):')
for fig_name in figures:
    path = os.path.join(results_dir, fig_name)
    status = 'ok' if os.path.exists(path) else 'MISSING'
    print(f'  [{status}] {fig_name}')
print(f'\nCSVs:')
print(f'  spectral_analysis_results.csv ({len(df_results)} filas)')
print(f'  per_emotion_results.csv ({len(df_emotions)} filas)')